# 🤖 Single Agent Pattern & Tool Calling — Step-by-Step Guide

**What is an Agent?**  
Standard Large Language Models (LLMs) operate strictly on static parametric knowledge memorized during pre-training. They cannot query live databases, fetch real-time product inventories, or trigger external APIs.

An **Agent** pairs an LLM with external **Tools** via an autonomous **ReAct (Reasoning + Action)** execution loop:
1. **Perceive & Reason**: The LLM analyzes the user prompt and reasons about what information is required.
2. **Select Action (Tool Call)**: If external data is needed, the LLM emits a structured tool invocation with specific function arguments.
3. **Execute & Observe**: The runtime environment executes the Python function and feeds the output back into the conversation context.
4. **Synthesize Response**: The LLM synthesizes the observed tool outputs into a natural, user-friendly final answer.

---

### Agent ReAct Architecture

```
┌────────────────────────────────────────────────────────┐
│                      User Request                      │
│      "What is the price and reviews of smart watch?"   │
└───────────────────────────┬────────────────────────────┘
                            │
                            ▼
┌────────────────────────────────────────────────────────┐
│               ReAct Agent Reasoning Loop               │
│                                                        │
│  1. THOUGHT: "I need product pricing and specs."       │
│  2. ACTION:  Call get_product("smart watch")           │
└─────────────┬────────────────────────────▲─────────────┘
              │ Tool Call                  │ Tool Output
              ▼                            │
┌───────────────────────────┐              │
│    get_product() Tool     │ ─────────────┘
│  Returns: Price, Specs    │
└───────────────────────────┘
              │
              ▼
┌────────────────────────────────────────────────────────┐
│               ReAct Agent Reasoning Loop               │
│                                                        │
│  3. THOUGHT: "Now I need customer review sentiment."   │
│  4. ACTION:  Call get_review("smart watch")            │
└─────────────┬────────────────────────────▲─────────────┘
              │ Tool Call                  │ Tool Output
              ▼                            │
┌───────────────────────────┐              │
│     get_review() Tool     │ ─────────────┘
│  Returns: Rating, Count   │
└───────────────────────────┘
              │
              ▼
┌────────────────────────────────────────────────────────┐
│                 Synthesized Final Answer               │
│ "The smartwatch costs $199.99 with 340 reviews (3.9★)" │
└────────────────────────────────────────────────────────┘
```

**Key Concepts Covered:**
1. **Function Schema Definition**: Using LangChain's `@tool` decorator to convert standard Python functions into LLM-callable schemas.
2. **Autonomous Tool Routing**: Providing the LLM with a list of tools so it can choose which tool to call based on user intent.
3. **Multi-Hop Tool Execution**: Executing sequential tool calls across a single user turn to combine multiple data sources.


---
## Step 1 — Setup & Environment Configuration

We import LangChain agent utilities:
- `ChatOpenAI`: The model interface (configured to use Google Gemini via the OpenAI-compatible endpoint or official OpenAI).
- `@tool`: Decorator to register Python functions as schema-backed tools.
- `create_agent`: Initializes the autonomous agent execution graph.


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent


load_dotenv()


True

---
## Step 2 — Product Knowledge Base & Tool Definition (`get_product`)

Here we create a simulated product database and expose it via the `get_product` tool.
> [!IMPORTANT]  
> The docstring `"""Look up a product by name and return its price, rating, stock, and description."""` is **not just documentation**; it is the functional description sent directly to the LLM to help it decide when and how to call this tool.


In [10]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name : str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found, Available: { ', '.join(PRODUCTS)}"
    return str(p)
    

---
## Step 3 — Creating the Single-Tool Agent

We initialize the agent with `create_agent`, providing:
1. **LLM**: The underlying reasoning model.
2. **Tools**: List of available functions `[get_product]`.
3. **System Prompt**: Defining the agent's role and tone.


In [13]:
    
    llm = ChatOpenAI(
        model="gpt-6-luna",
        # temperature=0,
    )
    
    agent = create_agent(
        llm,
        tools = [get_product], 
        system_prompt="You are a helpful product assistant for an online test store",
    )

---
## Step 4 — Query Execution & Output Formatting

The `ask` function invokes the agent and extracts clean text from the final message, stripping away raw tool calling metadata.


In [20]:
def get_text(content):
    if isinstance(content, list):
        return "\n".join(item["text"] for item in content if isinstance(item, dict) and item.get("type") == "text")
    return content

def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(get_text(result["messages"][-1].content))


In [21]:
ask("what is the price of wireless headphones")

Wireless headphones cost $79.99.


In [22]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a tenkeyless model with Cherry MX Brown switches and per-key RGB lighting. It’s rated 4.8/5 and costs $129.


---
## Step 5 — Expanding Tool Capability: Customer Reviews Tool (`get_review`)

To handle broader customer questions, we introduce a second data source and tool: `get_review`.
This tool returns review counts and customer satisfaction scores.


In [24]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}


@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

---
## Step 6 — Multi-Tool Agent Orchestration

We now construct `agent2` equipped with **both** `get_product` and `get_review`.
The LLM will dynamically select between them or call both in sequence based on what the user asks.


In [25]:

llm = ChatOpenAI(
    model="gpt-6-luna",
    # temperature=0,
)
    

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(get_text(result["messages"][-1].content))


---
## Step 7 — Testing Multi-Tool Queries & Multi-Hop Reasoning

Observe how:
1. When asked only about sentiment (`"how do people like smart watch"`), the agent calls only `get_review`.
2. When asked about both price and reviews (`"what is the price and reviews of smart watch"`), the agent performs multi-hop reasoning, invoking **both** `get_product` and `get_review` before synthesizing the answer!


In [26]:
ask2("how do people like smart watch")

The smartwatch is rated **3.9 out of 5** based on **340 reviews**—generally well liked, though opinions vary.


In [27]:
ask2("what is the price and reviews of smart watch")

The smart watch costs **$199.99**. It has **340 reviews** and a **3.9/5 review rating**. (The product listing separately shows a 4.3 rating.)
